In [1]:
from pyspark.sql import SparkSession
import spark


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
.appName ("DataFrame_Cache_Demo") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 15:40:17 INFO SparkEnv: Registering MapOutputTracker
26/04/14 15:40:17 INFO SparkEnv: Registering BlockManagerMaster
26/04/14 15:40:18 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/14 15:40:18 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
!hadoop fs -ls /tmp/

Found 7 items
-rw-r--r--   2 sowapipython hadoop   10528211 2026-04-01 18:47 /tmp/customers_10mb.csv
-rw-r--r--   2 sowapipython hadoop  168541068 2026-04-01 18:48 /tmp/customers_150mb.csv
-rw-r--r--   2 sowapipython hadoop    1060750 2026-04-01 18:47 /tmp/customers_1mb.csv
-rw-r--r--   2 sowapipython hadoop  343317147 2026-04-01 18:48 /tmp/customers_300mb.csv
drwxrwxrwt   - hdfs         hadoop          0 2026-03-31 14:46 /tmp/hadoop-yarn
drwx-wx-wx   - hive         hadoop          0 2026-03-31 14:47 /tmp/hive
-rw-r--r--   2 root         hadoop         84 2026-04-01 17:13 /tmp/inputhdfsdbz.txt


In [4]:
customers_df = spark.read \
               .option('header','true') \
               .csv('/tmp/customers_1mb.csv')

In [7]:
customers_df.rdd.getNumPartitions()

1

In [8]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



In [9]:
customers_df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [10]:
customers_df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



### Cache the data

In [11]:
customers_df.cache() # This behaviour can be different in different Platforms

DataFrame[customer_id: string, name: string, city: string, state: string, country: string, registration_date: string, is_active: string]

In [12]:
customers_df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [13]:
customers_df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [14]:
tail_df = customers_df.orderBy('customer_id', ascending=False)

In [15]:
tail_df.show(5)

+-----------+-------------+---------+-----------+-------+-----------------+---------+
|customer_id|         name|     city|      state|country|registration_date|is_active|
+-----------+-------------+---------+-----------+-------+-----------------+---------+
|       9999|Customer_9999|Hyderabad|  Karnataka|  India|       2023-06-02|    False|
|       9998|Customer_9998|     Pune|  Telangana|  India|       2023-01-27|    False|
|       9997|Customer_9997|Hyderabad|      Delhi|  India|       2023-06-05|     True|
|       9996|Customer_9996|    Delhi|  Telangana|  India|       2023-01-20|    False|
|       9995|Customer_9995|     Pune|Maharashtra|  India|       2023-03-18|     True|
+-----------+-------------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [16]:
customers_df.count()

17653

In [17]:
filtered_df = customers_df.filter(customers_df.city=='Mumbai')
filtered_df.show(5)

+-----------+-----------+------+-----------+-------+-----------------+---------+
|customer_id|       name|  city|      state|country|registration_date|is_active|
+-----------+-----------+------+-----------+-------+-----------------+---------+
|          9| Customer_9|Mumbai|  Telangana|  India|       2023-01-05|     True|
|         15|Customer_15|Mumbai|    Gujarat|  India|       2023-03-02|     True|
|         33|Customer_33|Mumbai|West Bengal|  India|       2023-09-11|    False|
|         60|Customer_60|Mumbai| Tamil Nadu|  India|       2023-12-01|     True|
|         63|Customer_63|Mumbai|  Karnataka|  India|       2023-05-23|     True|
+-----------+-----------+------+-----------+-------+-----------------+---------+
only showing top 5 rows



#### Uncache the data

In [18]:
customers_df.unpersist()

DataFrame[customer_id: string, name: string, city: string, state: string, country: string, registration_date: string, is_active: string]

In [19]:
filtered_df = customers_df.filter(customers_df.city=='Mumbai')
filtered_df.show(5)

+-----------+-----------+------+-----------+-------+-----------------+---------+
|customer_id|       name|  city|      state|country|registration_date|is_active|
+-----------+-----------+------+-----------+-------+-----------------+---------+
|          9| Customer_9|Mumbai|  Telangana|  India|       2023-01-05|     True|
|         15|Customer_15|Mumbai|    Gujarat|  India|       2023-03-02|     True|
|         33|Customer_33|Mumbai|West Bengal|  India|       2023-09-11|    False|
|         60|Customer_60|Mumbai| Tamil Nadu|  India|       2023-12-01|     True|
|         63|Customer_63|Mumbai|  Karnataka|  India|       2023-05-23|     True|
+-----------+-----------+------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [22]:
spark.stop()